# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [4]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [8]:
# 在这里编写你的训练代码
import torch

# 输入数据
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

# 1. 创建3个待学习参数，打开求导
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
w3 = torch.tensor(0.0, requires_grad=True)

# 优化器，把三个参数 w1,w2,w3 交给SGD
optimizer = torch.optim.SGD([w1, w2, w3], lr=0.02)

epochs = 5000
for epoch in range(epochs):
    optimizer.zero_grad()  # 清空上一轮梯度

    # 预测函数 ŷ = w1*x + w2*x² + w3
    y_pred = w1 * x + w2 * (x ** 2) + w3

    # 平均绝对损失（题目要求，不能用MSE）
    loss = torch.mean(torch.abs(y - y_pred))

    loss.backward()  # 自动求梯度
    optimizer.step() # 更新参数

    # 可选：每500轮打印一次损失观察收敛
    if epoch % 500 == 0:
        print(f"epoch {epoch:4d}, loss = {loss.item():.6f}")

# ========== 训练结束，输出结果 ==========
print("\n===== 学到的三个参数 =====")
print(f"w1 = {w1.item():.4f}")
print(f"w2 = {w2.item():.4f}")
print(f"w3 = {w3.item():.4f}")

# 用训练好的参数重新计算预测值
with torch.no_grad():
    y_pred_final = w1 * x + w2 * (x**2) + w3
    final_loss = torch.mean(torch.abs(y - y_pred_final))

print("\n===== 每个样本：x | 真实y | 预测ŷ =====")
for xi, yi, ypi in zip(x, y, y_pred_final):
    print(f"{xi.item():4.1f} | {yi.item():6.3f} | {ypi.item():6.3f}")

print(f"\n最终平均绝对误差loss = {final_loss.item():.6f}")


print(f"\n最终平均绝对误差loss = {final_loss.item():.6f}")


epoch    0, loss = 2.138889
epoch  500, loss = 0.034071
epoch 1000, loss = 0.034071
epoch 1500, loss = 0.034071
epoch 2000, loss = 0.034071
epoch 2500, loss = 0.034071
epoch 3000, loss = 0.034071
epoch 3500, loss = 0.034071
epoch 4000, loss = 0.034071
epoch 4500, loss = 0.034071

===== 学到的三个参数 =====
w1 = 1.5044
w2 = -0.5111
w3 = 1.9844

===== 每个样本：x | 真实y | 预测ŷ =====
-2.0 | -3.000 | -3.069
-1.5 | -1.375 | -1.422
-1.0 |  0.000 | -0.031
-0.5 |  1.125 |  1.104
 0.0 |  2.000 |  1.984
 0.5 |  2.625 |  2.609
 1.0 |  3.000 |  2.978
 1.5 |  3.125 |  3.091
 2.0 |  3.000 |  2.949

最终平均绝对误差loss = 0.034071

最终平均绝对误差loss = 0.034071


### 我的观察与说明
1：使用多个样本，损失不再是单个样本绝对值，而是全部样本绝对值的平均值；
2：优化器同时管理 3 个参数的梯度、同步更新。


## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。